# Assignment — Linear Regression on `penguins.csv`

**Question:** given a penguin's measurements, how heavy is it?

Target: `body_mass_g` &nbsp;·&nbsp; File: `penguins.csv`

Use only the numeric columns: `bill_length_mm`, `bill_depth_mm`, `flipper_length_mm`.

In [1]:
import numpy as np
import pandas as pd
df = pd.read_csv('../../Data/penguins.csv')
df.head()

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,Male
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,Female
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,Female
3,Adelie,Torgersen,NaN,NaN,NaN,NaN,NaN
4,Adelie,Torgersen,36.7,19.3,193.0,3450.0,Female


---
## 1. Look at the data

> **Flow:** Shape, columns, missing values.

Run `.info()`. Which columns have missing values, and how many?

In [2]:
df.info()
print(df.isna().sum())
# bill_length_mm, bill_depth_mm, flipper_length_mm, body_mass_g -> 2 missing each
# sex -> 11 missing

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 344 entries, 0 to 343
Data columns (total 7 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   species            344 non-null    object 
 1   island             344 non-null    object 
 2   bill_length_mm     342 non-null    float64
 3   bill_depth_mm      342 non-null    float64
 4   flipper_length_mm  342 non-null    float64
 5   body_mass_g        342 non-null    float64
 6   sex                333 non-null    object 
dtypes: float64(4), object(3)
memory usage: 18.9+ KB
species               0
island                0
bill_length_mm        2
bill_depth_mm         2
flipper_length_mm     2
body_mass_g           2
sex                  11
dtype: int64


---
## 2. Clean

> **Flow:** Keep the four numeric columns, drop the rows with blanks.

We cannot fill in `body_mass_g` — that is the answer we are trying to predict.
Rows missing it have to go.

Keep `bill_length_mm`, `bill_depth_mm`, `flipper_length_mm`, `body_mass_g`,
then `dropna()`.

In [3]:
cols = ['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g']
data = df[cols].dropna()

print(data.shape)

(342, 4)


*How many rows are left?*

**342 rows** are left (344 minus the 2 rows with blanks).

---
## 3. Features and target

In [4]:
features = ['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm']

X = data[features]
y = data['body_mass_g']

print(X.shape, y.shape)

(342, 3) (342,)


---
## 4. Split

> **Flow:** `test_size=0.2`, `random_state=42`.

In [5]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(X_train.shape, X_test.shape)

(273, 3) (69, 3)


---
## 5. One feature

> **Flow:** Start with `flipper_length_mm` alone.

Train `LinearRegression`. Print the coefficient, the intercept and the R².

In [6]:
from sklearn.linear_model import LinearRegression

one = LinearRegression()
one.fit(X_train[['flipper_length_mm']], y_train)

print("coefficient:", one.coef_[0])
print("intercept:", one.intercept_)
print("R2:", one.score(X_test[['flipper_length_mm']], y_test))

coefficient: 48.829682818993014
intercept: -5614.067120606901
R2: 0.7820354165340795


*Coefficient:* &nbsp;&nbsp; *R²:*

**Q.** The coefficient is about 48. Write one line explaining what that means in plain words —
what happens to the predicted weight if a penguin's flipper is 1 mm longer?

*Coefficient:* **48.83** &nbsp;&nbsp; *R²:* **0.782**

**Answer.** For every 1 mm longer flipper, the model predicts the penguin to be about **49 g heavier**.

---
## 6. All three features

> **Flow:** Same model, two more columns.

Print the R².

In [7]:
model = LinearRegression()
model.fit(X_train, y_train)

print("R2:", model.score(X_test, y_test))

R2: 0.7877806019338434


*R² with flipper only:* &nbsp;&nbsp; *R² with all three:*

**Q.** The score barely moved. That is not a mistake — it is the interesting part of this
assignment. Write two lines on why adding `bill_length_mm` and `bill_depth_mm` gave almost
nothing.

*Hint: what do all three columns really measure?*

**0.782** &nbsp;&nbsp; **0.788**

**Answer.** All three columns are really measuring the same thing — how big the penguin is. A long flipper already tells us it is a big penguin, and the bill measurements mostly repeat that information (bill length and flipper length are strongly correlated, 0.66), so there is very little new for the model to learn.

---
## 8. Metrics

> **Flow:** MAE, RMSE, R².

In [8]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

pred = model.predict(X_test)

mae = mean_absolute_error(y_test, pred)
rmse = np.sqrt(mean_squared_error(y_test, pred))
r2 = r2_score(y_test, pred)

print("MAE :", round(mae, 2))
print("RMSE:", round(rmse, 2))
print("R2  :", round(r2, 4))

MAE : 310.55
RMSE: 375.64
R2  : 0.7878


*MAE:* &nbsp;&nbsp; *RMSE:* &nbsp;&nbsp; *R²:*

**Q.** The average penguin weighs about 4200 g. Is your MAE large or small compared to that?
One line.

*MAE:* **310.5 g** &nbsp;&nbsp; *RMSE:* **375.6 g** &nbsp;&nbsp; *R²:* **0.788**

**Answer.** An error of about 310 g is roughly 7% of a 4200 g penguin — fairly small, the predictions are close but not exact.

---
## 8. Coefficients

In [9]:
# Print each feature name next to its coefficient
for name, coef in zip(features, model.coef_):
    print(name, ":", round(coef, 2))

bill_length_mm : 4.01
bill_depth_mm : 10.92
flipper_length_mm : 48.67


*Largest coefficient:*

**Q.** Does the largest coefficient mean that feature is the most important?
Careful — the three columns are measured on different scales. One line.

`flipper_length_mm` (**48.67**)

**Answer.** Not necessarily. A coefficient is "grams per 1 mm", and 1 mm means different things for each column (flipper length varies over ~60 mm, bill depth only ~8 mm). To compare importance fairly the features should be scaled first.

---
## 9. Questions

**Q1.** Why can we not use `accuracy_score` on this problem?

**Q2.** We dropped rows where `body_mass_g` was missing instead of filling them with the
median. Why is filling in the target a bad idea?

**Q3.** In class, adding more features to the mpg model raised R² from 0.723 to 0.824.
Here it barely changed. In two lines — what is different about these two datasets?

*Your answers:*

**Q1.** `accuracy_score` checks if the prediction is exactly equal to the answer. Weight is a continuous number, predicting exactly 4150.0 g almost never happens, so accuracy would be about 0. For regression we measure *how far off* we are (MAE, RMSE, R²).

**Q2.** The target is the answer the model learns from. If we fill it with the median, we are giving the model fake answers — it learns from values we made up, and on the test set we would be scoring it against made up values too.

**Q3.** In the mpg data the extra features (horsepower, weight, year, etc.) each added different information about fuel use. Here all three penguin features describe body size and overlap a lot, so once flipper length is in, the other two add almost nothing.